# Lab | Natural Language Processing
### SMS: SPAM or HAM

Completed exercises. Run cells in order; edit the detected column names if needed.

### Let’s prepare the environment

In [ ]:
import re
import string
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import MaxAbsScaler


- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speed up development.

## Read Data for the Fraudulent Email Kaggle Challenge

In [ ]:
data = pd.read_csv("data/kg_train.csv", encoding="latin-1")
# Reduce the training set for development; remove .head(1000) for final training.
data = data.head(1000).copy()
data.fillna("", inplace=True)
print("Shape:", data.shape)
print("Columns:", data.columns.tolist())
display(data.head())

### Let’s divide the training and test set into two partitions

In [ ]:
# Set these manually if your CSV uses different names.
text_candidates = ["text", "message", "email", "email_text", "body", "Message", "Text", "Email"]
target_candidates = ["label", "target", "class", "spam", "Label", "Category", "category", "is_spam"]
text_col = next((c for c in text_candidates if c in data.columns), None)
target_col = next((c for c in target_candidates if c in data.columns and c != text_col), None)
if text_col is None or target_col is None:
    raise ValueError(f"Set text_col and target_col to the correct CSV column names: {list(data.columns)}")
print("Text column:", text_col, "| Target column:", target_col)
print(data[target_col].value_counts())

# Split BEFORE fitting vectorizers to prevent test-set leakage.
class_counts = data[target_col].value_counts()
stratify_labels = data[target_col] if len(class_counts) > 1 and class_counts.min() >= 2 else None
data_train, data_val = train_test_split(
    data, test_size=0.2, random_state=42, stratify=stratify_labels
)
data_train = data_train.copy()
data_val = data_val.copy()
print("Train:", data_train.shape, "Validation:", data_val.shape)

## Data Preprocessing

In [ ]:
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

print(string.punctuation)
try:
    print(stopwords.words("english")[100:110])
except LookupError:
    print("NLTK stopwords not installed; run nltk.download('stopwords') if you need them.")
snowball = SnowballStemmer("english")

## Now, we have to clean the HTML code removing words

- First we remove inline JavaScript/CSS
- Then HTML comments
- Next remaining tags

In [ ]:
def strip_html(text):
    text = str(text)
    text = re.sub(r"(?is)<(script|style)\b[^>]*>.*?</\1\s*>", " ", text)
    text = re.sub(r"(?s)<!--.*?-->", " ", text)
    text = re.sub(r"(?s)<[^>]+>", " ", text)
    return text

data_train["html_clean_text"] = data_train[text_col].apply(strip_html)
data_val["html_clean_text"] = data_val[text_col].apply(strip_html)
display(data_train[[text_col, "html_clean_text"]].head(3))

## Clean the text

- Remove special characters, numbers, isolated single letters, extra spaces and prefixed `b`
- Convert to lowercase

In [ ]:
def clean_text(text):
    text = strip_html(text)
    # Handle a leading b prefix from string representations of byte literals.
    text = re.sub(r"^\s*b[\"']", "", text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b[a-z]\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data_train["clean_text"] = data_train["html_clean_text"].apply(clean_text)
data_val["clean_text"] = data_val["html_clean_text"].apply(clean_text)
display(data_train[["clean_text"]].head())

## Now let’s work on removing stopwords

In [ ]:
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    # Built-in fallback, so the notebook works without an NLTK data download.
    stop_words = set(ENGLISH_STOP_WORDS)

def remove_stopwords(text):
    return " ".join(word for word in text.split() if word not in stop_words)

data_train["no_stopwords"] = data_train["clean_text"].apply(remove_stopwords)
data_val["no_stopwords"] = data_val["clean_text"].apply(remove_stopwords)
display(data_train[["no_stopwords"]].head())

## Tame Your Text with Lemmatization

Reduce inflected words to dictionary forms (for example, *running* → *run*).

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

# WordNet needs to be installed once: in a notebook cell, run
# import nltk; nltk.download("wordnet")
try:
    lemmatizer.lemmatize("running", pos="v")
except LookupError as error:
    raise LookupError("NLTK WordNet data is missing. Run: import nltk; nltk.download('wordnet')") from error

def lemmatize_text(text):
    # Verb + noun passes provide straightforward lemmatization without POS tagging.
    return " ".join(lemmatizer.lemmatize(lemmatizer.lemmatize(word, pos="v"), pos="n")
                    for word in text.split())

data_train["preprocessed_text"] = data_train["no_stopwords"].apply(lemmatize_text)
data_val["preprocessed_text"] = data_val["no_stopwords"].apply(lemmatize_text)
display(data_train[[text_col, "preprocessed_text"]].head(3))

## Bag of Words

Find the top 10 words in HAM and SPAM messages (**exploratory data analysis**).

In [ ]:
labels = data_train[target_col].unique().tolist()
fig, axes = plt.subplots(1, len(labels), figsize=(7 * len(labels), 4), squeeze=False)
for ax, label in zip(axes.ravel(), labels):
    subset = data_train.loc[data_train[target_col] == label, "preprocessed_text"]
    counts = Counter(word for msg in subset for word in msg.split())
    top = counts.most_common(10)
    if top:
        words, frequencies = zip(*top)
        ax.barh(list(words)[::-1], list(frequencies)[::-1])
    ax.set_title(f"Top 10 words: {label}")
    ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

## Extra features

Add money-related words, suspicious words, and text length.

In [ ]:
money_simbol_list = "|".join(["euro", "dollar", "pound", "€", r"\$"])
suspicious_words = "|".join(["free", "cheap", "sex", "money", "account", "bank", "fund", "transfer", "transaction", "win", "deposit", "password"])

for frame in (data_train, data_val):
    # Raw text retains €/$ signs that were discarded by alphabetic cleaning.
    frame["money_mark"] = frame[text_col].astype(str).str.contains(
        money_simbol_list, case=False, regex=True, na=False
    ).astype(int)
    frame["suspicious_words"] = frame["preprocessed_text"].str.contains(
        suspicious_words, case=False, regex=True, na=False
    ).astype(int)
    frame["text_len"] = frame["preprocessed_text"].str.len()

display(data_train[["preprocessed_text", "money_mark", "suspicious_words", "text_len"]].head())

## How would Bag of Words work with CountVectorizer?

In [ ]:
# Fit on training text only; transform validation text with the same vocabulary.
count_vectorizer = CountVectorizer()
X_train_bow = count_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_bow = count_vectorizer.transform(data_val["preprocessed_text"])
print("BoW train matrix:", X_train_bow.shape)
print("BoW validation matrix:", X_val_bow.shape)
print("Sample vocabulary:", count_vectorizer.get_feature_names_out()[:20])

## TF-IDF

- Load vectorizer
- Vectorize dataset
- Print shape

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf_vectorizer.transform(data_val["preprocessed_text"])
print("TF-IDF train matrix:", X_train_tfidf.shape)
print("TF-IDF validation matrix:", X_val_tfidf.shape)

## And then train a classifier?

The required classifier is `MultinomialNB()` with **default parameters**.

In [ ]:
y_train = data_train[target_col]
y_val = data_val[target_col]

baseline_model = MultinomialNB()
baseline_model.fit(X_train_tfidf, y_train)
baseline_predictions = baseline_model.predict(X_val_tfidf)
print("TF-IDF accuracy:", accuracy_score(y_val, baseline_predictions))
print(classification_report(y_val, baseline_predictions, zero_division=0))
print("Confusion matrix:", confusion_matrix(y_val, baseline_predictions))

### Extra Task – Implement a SPAM/HAM classifier

Compare **BoW**, **TF-IDF**, **BoW + flags**, and **TF-IDF + flags**. Keep `MultinomialNB()` at its default settings. Use validation data to select the best features.

In [ ]:
extra_cols = ["money_mark", "suspicious_words", "text_len"]
# Non-negative numeric features are required by MultinomialNB.
# Scale the three extra features based on training data only.
extra_scaler = MaxAbsScaler()
extra_train = extra_scaler.fit_transform(data_train[extra_cols].to_numpy(dtype=float))
extra_val = extra_scaler.transform(data_val[extra_cols].to_numpy(dtype=float))

X_train_bow_extra = hstack([X_train_bow, csr_matrix(extra_train)], format="csr")
X_val_bow_extra = hstack([X_val_bow, csr_matrix(extra_val)], format="csr")
X_train_tfidf_extra = hstack([X_train_tfidf, csr_matrix(extra_train)], format="csr")
X_val_tfidf_extra = hstack([X_val_tfidf, csr_matrix(extra_val)], format="csr")

options = {
    "Bag of Words": (X_train_bow, X_val_bow),
    "TF-IDF": (X_train_tfidf, X_val_tfidf),
    "Bag of Words + extra features": (X_train_bow_extra, X_val_bow_extra),
    "TF-IDF + extra features": (X_train_tfidf_extra, X_val_tfidf_extra),
}

results = []
models = {}
for name, (Xtr, Xva) in options.items():
    model = MultinomialNB()  # unchanged default classifier
    model.fit(Xtr, y_train)
    pred = model.predict(Xva)
    results.append({
        "Features": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Macro F1": f1_score(y_val, pred, average="macro", zero_division=0),
    })
    models[name] = model

results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
display(results_df)
print("Best by macro F1:", results_df.iloc[0]["Features"])
# Macro F1 is helpful when spam/ham classes are imbalanced.
# For the Kaggle submission, retrain the chosen setup on all labeled training data,
# then transform the Kaggle test data with the same fitted preprocessing and vectorizer.

**Review notes:** Results depend on the dataset and split. If you need competition predictions, use the winning feature set, retrain on all available labeled training rows, and follow the submission format provided by the competition.